# 01 - Xử lý bảng CUSTOMER (chuẩn 3NF, tiêu chuẩn Silver)

**Nguồn dữ liệu:** `customer_silver.csv`
**Bảng đích:** `CUSTOMER` theo lược đồ 3NF (`Luoc_do_quan_he_3NF.docx`, mục 3)

| Thuộc tính đích | Nguồn | Ghi chú |
|---|---|---|
| customer_id (PK) | customer_id | giữ nguyên |
| zip (FK -> GEOGRAPHY) | zip | giữ nguyên |
| signup_date | signup_date | parse về kiểu date |
| gender | gender | chuẩn hóa chuỗi |
| age_group | age_group | chuẩn hóa chuỗi |
| acquisition_channel | acquisition_channel | chuẩn hóa chuỗi |

**Cột bị loại bỏ:** `city` — dư thừa vì `customer_id -> zip -> city` (phụ thuộc bắc cầu đã được xử lý trong lược đồ 3NF, `city` thuộc về bảng `GEOGRAPHY`).

Notebook này gồm 3 phần: (1) Nạp dữ liệu & khảo sát chất lượng, (2) Làm sạch & chuyển đổi theo lược đồ 3NF, (3) Kiểm tra ràng buộc & xuất file Silver chuẩn hóa.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ==== Cấu hình đường dẫn (chỉnh lại nếu chạy ở môi trường khác) ====
RAW_DIR = Path("./DAAI_N1.4/silver_data_raw")          # nơi chứa các file *_silver.csv gốc
OUTPUT_DIR = Path("./DAAI_N1.4/silver_data")  # nơi xuất file bảng 3NF đã chuẩn hóa
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SRC_FILE = RAW_DIR / "customer_silver.csv"
OUT_FILE = OUTPUT_DIR / "CUSTOMER.csv"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

## 1. Nạp dữ liệu & khảo sát chất lượng (Data Profiling)

In [2]:
df_raw = pd.read_csv(SRC_FILE)
print("Shape:", df_raw.shape)
df_raw.head()

Shape: (121930, 7)


,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media
1,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign
2,3,15201,Hai Phong,2018-07-24,Female,18-24,organic_search
3,4,15201,Hai Phong,2017-11-29,Male,35-44,referral
4,5,15201,Hai Phong,2022-09-23,Male,55+,organic_search


In [3]:
# Kiểu dữ liệu & số lượng null
display(df_raw.dtypes)
print()
print("Null theo cột:")
display(df_raw.isnull().sum())

customer_id             int64
zip                     int64
city                   object
signup_date            object
gender                 object
age_group              object
acquisition_channel    object
dtype: object


Null theo cột:


customer_id            0
zip                    0
city                   0
signup_date            0
gender                 0
age_group              0
acquisition_channel    0
dtype: int64

In [4]:
# Kiểm tra khóa chính customer_id: không được trùng, không null
n_dup_pk = df_raw['customer_id'].duplicated().sum()
n_null_pk = df_raw['customer_id'].isnull().sum()
print(f"customer_id trùng lặp: {n_dup_pk}")
print(f"customer_id null: {n_null_pk}")
assert n_dup_pk == 0, "Vi phạm khóa chính: customer_id bị trùng!"
assert n_null_pk == 0, "Vi phạm khóa chính: customer_id có giá trị null!"

customer_id trùng lặp: 0
customer_id null: 0


In [5]:
# Miền giá trị các cột phân loại (categorical) -> phát hiện lỗi chính tả / khoảng trắng thừa
for col in ['gender', 'age_group', 'acquisition_channel']:
    print(f"--- {col} ---")
    print(sorted(df_raw[col].unique()))
    print()

--- gender ---
['Female', 'Male', 'Non-binary']

--- age_group ---
['18-24', '25-34', '35-44', '45-54', '55+']

--- acquisition_channel ---
['direct', 'email_campaign', 'organic_search', 'paid_search', 'referral', 'social_media']



In [6]:
# Kiểm tra định dạng ngày và khoảng giá trị zip
invalid_dates = pd.to_datetime(df_raw['signup_date'], errors='coerce').isna().sum()
print(f"signup_date không parse được: {invalid_dates}")
print(f"zip: min={df_raw['zip'].min()}, max={df_raw['zip'].max()}, số zip duy nhất={df_raw['zip'].nunique()}")
print(f"Số dòng trùng lặp hoàn toàn (full duplicate rows): {df_raw.duplicated().sum()}")

signup_date không parse được: 0
zip: min=1001, max=99950, số zip duy nhất=31491
Số dòng trùng lặp hoàn toàn (full duplicate rows): 0


**Nhận xét khảo sát:**
- Không có giá trị null ở bất kỳ cột nào.
- `customer_id` là khóa duy nhất, không trùng lặp.
- Các cột phân loại (`gender`, `age_group`, `acquisition_channel`) có miền giá trị hợp lệ, nhất quán, không phát hiện lỗi chính tả/viết hoa-thường.
- `signup_date` parse hợp lệ 100%.
- `city` sẽ được loại khỏi bảng đích vì thuộc về `GEOGRAPHY` theo lược đồ 3NF.

## 2. Làm sạch & chuyển đổi theo lược đồ 3NF

In [7]:
df = df_raw.copy()

# (a) Loại bỏ cột dư thừa theo lược đồ 3NF (city thuộc GEOGRAPHY, suy ra được qua zip)
df = df.drop(columns=['city'])

# (b) Chuẩn hóa chuỗi: loại khoảng trắng thừa hai đầu, tránh lỗi khi group/join về sau
str_cols = ['gender', 'age_group', 'acquisition_channel']
for col in str_cols:
    df[col] = df[col].str.strip()

# (c) Ép kiểu dữ liệu đúng theo lược đồ
df['customer_id'] = df['customer_id'].astype('int64')
df['zip'] = df['zip'].astype('int64')
df['signup_date'] = pd.to_datetime(df['signup_date']).dt.date

# (d) Loại bỏ dòng trùng lặp hoàn toàn (nếu có) — phòng vệ, dữ liệu hiện tại không có
before = len(df)
df = df.drop_duplicates()
print(f"Loại bỏ {before - len(df)} dòng trùng lặp hoàn toàn.")

# (e) Sắp xếp lại thứ tự cột đúng theo lược đồ CUSTOMER
df = df[['customer_id', 'zip', 'signup_date', 'gender', 'age_group', 'acquisition_channel']]

df.head()

Loại bỏ 0 dòng trùng lặp hoàn toàn.


,customer_id,zip,signup_date,gender,age_group,acquisition_channel
0,1,15201,2021-12-30,Female,35-44,social_media
1,2,15201,2013-12-27,Female,45-54,email_campaign
2,3,15201,2018-07-24,Female,18-24,organic_search
3,4,15201,2017-11-29,Male,35-44,referral
4,5,15201,2022-09-23,Male,55+,organic_search


## 3. Kiểm tra ràng buộc cuối & xuất bảng Silver chuẩn hóa

In [8]:
# Kiểm tra ràng buộc khóa chính lần cuối trên bảng đã xử lý
assert df['customer_id'].is_unique, "customer_id không còn là khóa duy nhất sau xử lý!"
assert df['customer_id'].notnull().all()
assert df['zip'].notnull().all()
assert df['signup_date'].notnull().all()

print("Số dòng cuối cùng:", len(df))
print("Số cột:", len(df.columns))
print()
print("Kiểu dữ liệu cuối cùng:")
display(df.dtypes)

Số dòng cuối cùng: 121930
Số cột: 6

Kiểu dữ liệu cuối cùng:


customer_id             int64
zip                     int64
signup_date            object
gender                 object
age_group              object
acquisition_channel    object
dtype: object

In [9]:
df.to_csv(OUT_FILE, index=False)
print(f"Đã xuất bảng CUSTOMER (chuẩn Silver, 3NF) tại: {OUT_FILE.resolve()}")
print(f"Số dòng: {len(df):,} | Số cột: {len(df.columns)}")

Đã xuất bảng CUSTOMER (chuẩn Silver, 3NF) tại: D:\TH_DA&AI\DAAI_N1.4\silver_data\CUSTOMER.csv
Số dòng: 121,930 | Số cột: 6
